In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import qmc
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, mean_absolute_error

### Function Description

You’re optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown. 

Your objective is to find the parameter combination that maximises the function’s output, such as performance, efficiency or validation accuracy. Because the function is high-dimensional and likely complex, global optimisation is hard, so identifying strong local maxima is often a practical strategy.

For example, imagine you’re tuning an ML model with eight hyperparameters: learning rate, batch size, number of layers, dropout rate, regularisation strength, activation function (numerically encoded), optimiser type (encoded) and initial weight range. Each input set returns a single validation accuracy score between 0 and 1. Your goal is to maximise this score.

In [2]:
# Load data
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy').ravel()

In [3]:
df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']) 
df['target'] = Y

print(df.describe())

              x1         x2         x3         x4         x5         x6  \
count  40.000000  40.000000  40.000000  40.000000  40.000000  40.000000   
mean    0.534609   0.472150   0.515970   0.430449   0.468951   0.460126   
std     0.310676   0.309706   0.282440   0.261479   0.281380   0.277376   
min     0.009077   0.003419   0.022929   0.009043   0.009649   0.022113   
25%     0.302376   0.206189   0.306074   0.179607   0.236969   0.245246   
50%     0.532770   0.540389   0.488684   0.470111   0.411933   0.409715   
75%     0.792979   0.712416   0.732348   0.663788   0.722449   0.706965   
max     0.985945   0.973980   0.998885   0.902986   0.986902   0.990244   

              x7         x8     target  
count  40.000000  40.000000  40.000000  
mean    0.579196   0.506720   7.815274  
std     0.257913   0.281347   0.958966  
min     0.035909   0.041956   5.592193  
25%     0.428796   0.294756   7.257810  
50%     0.596949   0.525116   7.889150  
75%     0.770316   0.729496   8.49013

In [4]:
# Kernel Setup (8 Dimensions)
# normalize_y=True ensures stability across the massive 8D voids
initial_length_scales = np.array([1.0] * 8)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=50,  
    random_state=42
)

model.fit(X, Y)

# 8D Monte Carlo Random Search Space
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 8))

mean, std = model.predict(search_grid, return_std=True)

# Acquisition Function (UCB Beta Sweep)
betas = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 
        'x5': bp[4], 'x6': bp[5], 'x7': bp[6], 'x8': bp[7],
        'Pred Mean': mean[best_idx],
        'Uncert': std[best_idx]
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
# Setting pandas to show all columns so it doesn't truncate the 8D output
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
  Beta     x1     x2     x3     x4     x5     x6     x7     x8  Pred Mean  Uncert
0.1000 0.2307 0.2916 0.1946 0.1403 0.9801 0.6136 0.2976 0.7945     9.9439  0.1627
0.2500 0.1038 0.1739 0.1453 0.2175 0.9923 0.5895 0.2032 0.5255     9.9384  0.2043
0.5000 0.2341 0.0168 0.2491 0.0460 0.9457 0.8903 0.1609 0.5881     9.9302  0.2314
0.7500 0.2341 0.0168 0.2491 0.0460 0.9457 0.8903 0.1609 0.5881     9.9302  0.2314
1.0000 0.2228 0.0663 0.1140 0.0623 0.9654 0.7060 0.0778 0.1581     9.8958  0.2731
1.5000 0.2228 0.0663 0.1140 0.0623 0.9654 0.7060 0.0778 0.1581     9.8958  0.2731
2.0000 0.2228 0.0663 0.1140 0.0623 0.9654 0.7060 0.0778 0.1581     9.8958  0.2731
3.0000 0.1552 0.0291 0.2629 0.0510 0.8790 0.6365 0.0060 0.2408     9.8287  0.3040
4.0000 0.1552 0.0291 0.2629 0.0510 0.8790 0.6365 0.0060 0.2408     9.8287  0.3040
5.0000 0.0043 0.2783 0.0085 0.9736 0.9950 0.7162 0.0746 0.1248     9.4285  0.4026

--- Optimized Kernel Parameters ---
Matern(lengt

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 1 submission: x1 = 0.2228, x2 = 0.0663, x3 = 0.1140, x4 = 0.0623, x5 = 0.9654, x6 = 0.7060, x7 = 0.0778, x8 = 0.1581

* The Problem Framing: Optimizing an 8-dimensional black-box function with a significant local peak at Index 14 (score of 9.598).

* The Surrogate Model: Matern 2.5 + 8D ARD.
    * normalize_y=True: Applied to stabilize the model across the vast 8D space, preventing the optimizer from becoming "paralyzed" by the empty space between samples.
    * High Restarts (50): Utilized to ensure the log-marginal-likelihood surface was thoroughly searched for the most accurate ARD length scales.

* Key Insights from the Model: The "Dummy" Variables (x6​ and x8​): The length scales for these two dimensions hit the maximum bound (10.0), effectively proving they have no impact on the performance score. This reduced the functional complexity from 8D to 6D.

* The Sensitive Core (x3​): With a length scale of 0.802, x3​ was identified as the most critical parameter for maintaining the local peak.

* The Final Query (Beta = 1.0): [0.2228, 0.0663, 0.1140, 0.0623, 0.9654, 0.7060, 0.0778, 0.1581]
    * The Logic: This coordinate represents the "Resilient Block." While Beta 0.5 targeted a slightly higher predicted mean, Beta 1.0 targets a wide, stable plateau. It maintains a more conservative stance on the sensitive x3​ knob while still pushing for a predicted score of 9.89.

In [5]:
# ---------------------------------------------------
# WEEK1: NEW DATA HERE
# ---------------------------------------------------
w1_new_input = [0.2228, 0.0663, 0.114 , 0.0623, 0.9654, 0.706 , 0.0778, 0.1581]
w1_new_output = 9.848867519

new_X = np.array(w1_new_input).reshape(1, -1)
new_Y = np.atleast_1d(w1_new_output)

X_updated_w1 = np.vstack((X, new_X))
Y_updated_w1 = np.append(Y, new_Y)

df = pd.DataFrame(X_updated_w1, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w1

print(df.describe())

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  41.000000  41.000000  41.000000  41.000000  41.000000  41.000000  41.000000  41.000000  41.000000
mean    0.527004   0.462251   0.506165   0.421470   0.481059   0.466123   0.566967   0.498217   7.864874
std     0.310609   0.312309   0.285865   0.264514   0.288456   0.276565   0.266435   0.283093   0.998745
min     0.009077   0.003419   0.022929   0.009043   0.009649   0.022113   0.035909   0.041956   5.592193
25%     0.222800   0.168203   0.281869   0.177981   0.240298   0.245548   0.410153   0.292472   7.299872
50%     0.532721   0.523642   0.481372   0.452656   0.414579   0.410105   0.592415   0.524885   7.923759
75%     0.791888   0.699506   0.729546   0.662216   0.730693   0.706650   0.765670   0.728611   8.541748
max     0.985945   0.973980   0.998885   0.902986   0.986902   0.990244   0.992914   0.988755   9.848868


In [6]:
initial_length_scales = np.array([1.0] * 8)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 20.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=50,  
    random_state=42
)

model.fit(X_updated_w1, Y_updated_w1)

# 8D Monte Carlo Random Search Space
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 8))

mean, std = model.predict(search_grid, return_std=True)

# Acquisition Function (UCB Beta Sweep)
betas = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 
        'x5': bp[4], 'x6': bp[5], 'x7': bp[6], 'x8': bp[7],
        'Pred Mean': mean[best_idx],
        'Uncert': std[best_idx]
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
# Setting pandas to show all columns so it doesn't truncate the 8D output
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5       x6       x7       x8  Pred Mean   Uncert
0.100000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
0.250000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
0.500000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
0.750000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
1.000000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
1.500000 0.230660 0.291570 0.194562 0.140270 0.980146 0.613562 0.297646 0.794540   9.928947 0.114669
2.000000 0.032442 0.493511 0.032259 0.455361 0.902644 0.497971 0.043311 0.264370   9.618178 0.286041
3.000000 0.004318 0.278304 0.008528 0.973578 0.995037 0.716242 0.074606 0.124800   9.387755 0.382020

--- Optimized Kernel Parameters ---
Mater

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 2 Submission:

* **The Observation**: Beta sweep reveals a **bifurcation point** at β=1.5 → β=2.0. Betas [0.1–1.5] consistently suggest the same point, while β≥2.0 explore more aggressively into higher-uncertainty regions.

* **The Decision**: Accept the consensus point suggested across β=[0.1–1.5]: **[0.2307, 0.2916, 0.1946, 0.1403, 0.9801, 0.6136, 0.2976, 0.7945]** with predicted mean ≈ 9.929.

* **Rationale**: This point balances exploitation (leveraging the high-confidence peak) with robustness. The stability across multiple beta values signals genuine model confidence rather than noise.

In [7]:
# ---------------------------------------------------
# WEEK2: NEW DATA HERE
# ---------------------------------------------------

w2_new_input = [0.230660, 0.291570, 0.194562, 0.140270, 0.980146, 0.613562, 0.297646, 0.794540]
w2_new_output = np.float64(9.881237753174)

w2_new_X = np.atleast_2d(w2_new_input)
w2_new_Y = np.atleast_1d(w2_new_output)

X_updated_w2 = np.vstack((X_updated_w1, w2_new_X))
Y_updated_w2 = np.append(Y_updated_w1, w2_new_Y)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_updated_w2_scaled = scaler_x.fit_transform(X_updated_w2)
Y_updated_w2_scaled = scaler_y.fit_transform(Y_updated_w2.reshape(-1, 1)).flatten()

df = pd.DataFrame(X_updated_w2, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w2

print(df.describe())
# print(df.head(42))

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  42.000000  42.000000  42.000000  42.000000  42.000000  42.000000  42.000000  42.000000  42.000000
mean    0.519948   0.458187   0.498746   0.414774   0.492942   0.469633   0.560554   0.505273   7.912883
std     0.310187   0.309599   0.286422   0.264847   0.295141   0.274117   0.266427   0.283333   1.034392
min     0.009077   0.003419   0.022929   0.009043   0.009649   0.022113   0.035909   0.041956   5.592193
25%     0.224765   0.180865   0.272628   0.155065   0.242101   0.247226   0.335271   0.293234   7.302627
50%     0.519375   0.503188   0.451130   0.438626   0.447599   0.427611   0.591723   0.525116   7.940817
75%     0.788620   0.688062   0.729167   0.656450   0.733288   0.706488   0.764873   0.731268   8.590025
max     0.985945   0.973980   0.998885   0.902986   0.986902   0.990244   0.992914   0.988755   9.881238


In [8]:
initial_length_scales = np.array([1.0] * 8)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 50.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-7, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model.fit(X_updated_w2_scaled, Y_updated_w2_scaled)

def acquisition_function(x_trial, beta, model):
    x_trial = x_trial.reshape(1, -1)
    mean, std = model.predict(x_trial, return_std=True)
    ucb = mean[0] + (beta * std[0])
    return -float(ucb)

betas = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

sampler = qmc.Sobol(d=8, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'x7': best_x_raw[6],
        'x8': best_x_raw[7],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
# Setting pandas to show all columns so it doesn't truncate the 8D output
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-07. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5       x6       x7       x8  Pred Mean  Uncertainty
0.100000 0.186904 0.149290 0.189967 0.059927 1.000000 1.000000 0.235004 0.257347   9.920886     0.063525
0.250000 0.188288 0.139242 0.196905 0.042261 1.000000 1.000000 0.239275 0.000000   9.918028     0.081048
0.500000 0.192823 0.109446 0.216005 0.000000 1.000000 1.000000 0.240193 0.000000   9.911997     0.097238
0.750000 0.199434 0.079035 0.237360 0.000000 1.000000 1.000000 0.234794 0.000000   9.904226     0.109594
1.000000 0.205120 0.043871 0.260130 0.000000 1.000000 1.000000 0.221104 0.000000   9.891298     0.124306
1.500000 0.000000 0.362335 0.000000 0.371721 1.000000 1.000000 0.082527 1.000000   9.711847     0.278683
2.000000 0.000000 0.426734 0.000000 0.501529 1.000000 1.000000 0.000000 1.000000   9.613836     0.335009
3.000000 0.000000 0.481066 0.000000 0.875249 1.000000 1.000000 0.000000 1.000000   9.414747     0.413404

--- O

### Week 3 Submission:

* Changed the whole approach of the search function, standardized the whole set of features and output. 

* Next query: β=[0.50]: **[0.192823 , 0.109446 , 0.216005 , 0.000000 , 1.000000 , 1.000000 , 0.240193 , 0.000000]** with predicted mean ≈ 9.912.

In [9]:
# ---------------------------------------------------
# WEEK3: NEW DATA HERE
# ---------------------------------------------------

w3_new_input = 	[0.192823, 0.109446, 0.216005, 0.000000, 1.000000, 1.000000, 0.240193, 0.000000]
w3_new_output = 9.627201619853

w3_new_X = np.atleast_2d(w3_new_input)
w3_new_Y = np.atleast_1d(w3_new_output)

X_updated_w3 = np.vstack((X_updated_w2, w3_new_X))
Y_updated_w3 = np.append(Y_updated_w2, w3_new_Y)

X_updated_w3_scaled = scaler_x.fit_transform(X_updated_w3)
Y_updated_w3_scaled = scaler_y.fit_transform(Y_updated_w3.reshape(-1, 1))

df = pd.DataFrame(X_updated_w3, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w3

print(df.describe())

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  43.000000  43.000000  43.000000  43.000000  43.000000  43.000000  43.000000  43.000000  43.000000
mean    0.512340   0.450077   0.492171   0.405128   0.504734   0.481967   0.553104   0.493522   7.952751
std     0.310505   0.310480   0.286258   0.269211   0.301684   0.282653   0.267731   0.290351   1.054911
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.022113   0.035909   0.000000   5.592193
25%     0.207811   0.143765   0.263873   0.143848   0.243904   0.248903   0.303978   0.249157   7.305383
50%     0.506028   0.482734   0.420888   0.424597   0.480619   0.445117   0.591030   0.524885   7.957875
75%     0.785353   0.676618   0.728788   0.650684   0.735779   0.707279   0.764076   0.730382   8.711838
max     0.985945   0.973980   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.881238


In [10]:
initial_length_scales = np.array([1.0] * 8)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-15, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model.fit(X_updated_w3_scaled, Y_updated_w3_scaled)

betas = [0.01, 0.05, 0.10, 0.15, 0.35, 0.5, 0.75]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'x7': best_x_raw[6],
        'x8': best_x_raw[7],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
# Setting pandas to show all columns so it doesn't truncate the 8D output
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5       x6       x7       x8  Pred Mean  Uncertainty
0.010000 0.139771 0.090214 0.176581 0.000000 1.000000 0.355379 0.167334 0.685490  10.020022     0.123455
0.050000 0.137949 0.086552 0.177681 0.000000 1.000000 0.349890 0.165028 0.685476  10.019947     0.125953
0.100000 0.135606 0.081870 0.179051 0.000000 1.000000 0.342978 0.162020 0.685480  10.019708     0.129137
0.150000 0.133183 0.077066 0.180408 0.000000 1.000000 0.335992 0.158859 0.685480  10.019300     0.132397
0.350000 0.122528 0.056558 0.185595 0.000000 1.000000 0.307260 0.144547 0.685479  10.015803     0.146287
0.500000 0.113221 0.039618 0.189036 0.000000 1.000000 0.284645 0.131781 0.685496  10.010900     0.157792
0.750000 0.093753 0.007467 0.193123 0.000000 1.000000 0.243869 0.105325 0.685500   9.996901     0.180034

--- Optimized Kernel Parameters ---
Matern(length_scale=[4.57, 6.7, 3.54, 9.61, 16.9, 7.77, 5.43, 1e+04], nu=2

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 4 submission

Next point to query [0.135606 , 0.081870 , 0.179051 , 0.000000 , 1.000000 , 0.342978 , 0.162020 , 0.685480] beta = 0.10 

* Found an impressive improvement in the last run, optimise it further 
* Consider running NNs for the next round in parallel

In [11]:
# ---------------------------------------------------
# WEEK4: NEW DATA HERE
# ---------------------------------------------------

w4_new_input = 	[0.135606, 0.081870, 0.179051, 0.000000, 1.000000, 0.342978, 0.162020, 0.685480]
w4_new_output = 9.914833174501

w4_new_X = np.atleast_2d(w4_new_input)
w4_new_Y = np.atleast_1d(w4_new_output)

X_updated_w4 = np.vstack((X_updated_w3, w4_new_X))
Y_updated_w4 = np.append(Y_updated_w3, w4_new_Y)

X_updated_w4_scaled = scaler_x.fit_transform(X_updated_w4)
Y_updated_w4_scaled = scaler_y.fit_transform(Y_updated_w4.reshape(-1, 1))

df = pd.DataFrame(X_updated_w4, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w4

print(df.describe())

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  44.000000  44.000000  44.000000  44.000000  44.000000  44.000000  44.000000  44.000000  44.000000
mean    0.503778   0.441708   0.485055   0.395921   0.515990   0.478808   0.544216   0.497885   7.997344
std     0.312085   0.311829   0.286821   0.272982   0.307362   0.280132   0.271089   0.288410   1.083721
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.022113   0.035909   0.000000   5.592193
25%     0.192777   0.116857   0.253214   0.138801   0.245706   0.250581   0.292295   0.270814   7.308138
50%     0.493637   0.470746   0.418842   0.406103   0.491909   0.427611   0.574163   0.525116   7.967780
75%     0.782086   0.665174   0.728409   0.644918   0.738183   0.706965   0.763280   0.729496   8.820855
max     0.985945   0.973980   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.914833


In [12]:
initial_length_scales = np.array([1.0] * 8)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-20, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model.fit(X_updated_w4_scaled, Y_updated_w4_scaled)

betas = [0.01, 0.05, 0.10, 0.15, 0.35, 0.5, 0.75]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'x7': best_x_raw[6],
        'x8': best_x_raw[7],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
# Setting pandas to show all columns so it doesn't truncate the 8D output
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5       x6       x7       x8  Pred Mean  Uncertainty
0.010000 0.164574 0.157848 0.140142 0.127107 1.000000 0.471799 0.204873 0.302501   9.948624     0.033972
0.050000 0.160750 0.157527 0.136521 0.133009 1.000000 0.473368 0.205111 0.302501   9.948534     0.036942
0.100000 0.155373 0.157175 0.131683 0.140679 1.000000 0.475615 0.205260 0.302501   9.948222     0.041070
0.150000 0.149413 0.156911 0.126539 0.148590 1.000000 0.478105 0.205171 0.302500   9.947652     0.045620
0.350000 0.120813 0.157007 0.103519 0.181135 1.000000 0.489986 0.202154 0.688610   9.942138     0.067318
0.500000 0.094802 0.158450 0.083442 0.205947 1.000000 0.500243 0.196447 0.688610   9.933611     0.087278
0.750000 0.040480 0.163072 0.041082 0.248232 1.000000 0.519548 0.177833 0.204497   9.906037     0.130848

--- Optimized Kernel Parameters ---
Matern(length_scale=[4.59, 6.83, 3.65, 9.14, 16.8, 8.11, 5.55, 1e+05], nu=

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 5 submission

Next point to query [0.164574 0.157848 0.140142 0.127107 1.000000 0.471799 0.204873 0.302501] beta = 0.01

* Found an improvement, but it shows signs of diminishment - possible max in this area
* If there's no improvement, switch to a different strategy - exploration or neural network approach

In [13]:
# ---------------------------------------------------
# WEEK5: NEW DATA HERE
# ---------------------------------------------------

w5_new_input = 	[0.164574, 0.157848, 0.140142, 0.127107, 1.000000, 0.471799, 0.204873, 0.302501]
w5_new_output = 9.9610727818439

w5_new_X = np.atleast_2d(w5_new_input)
w5_new_Y = np.atleast_1d(w5_new_output)

X_updated_w5 = np.vstack((X_updated_w4, w5_new_X))
Y_updated_w5 = np.append(Y_updated_w4, w5_new_Y)

X_updated_w5_scaled = scaler_x.fit_transform(X_updated_w5)
Y_updated_w5_scaled = scaler_y.fit_transform(Y_updated_w5.reshape(-1, 1))

df = pd.DataFrame(X_updated_w5, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w5

print(df.describe())

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  45.000000  45.000000  45.000000  45.000000  45.000000  45.000000  45.000000  45.000000  45.000000
mean    0.496240   0.435400   0.477390   0.389947   0.526746   0.478653   0.536675   0.493543   8.040982
std     0.312634   0.311156   0.288167   0.272822   0.312298   0.276932   0.272723   0.286598   1.110610
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.022113   0.035909   0.000000   5.592193
25%     0.192640   0.119328   0.238260   0.134396   0.247509   0.252259   0.276241   0.292472   7.310894
50%     0.481245   0.458759   0.416796   0.387609   0.503199   0.445117   0.557295   0.524885   7.977685
75%     0.778818   0.653730   0.728030   0.639151   0.740520   0.706650   0.762483   0.728611   8.830745
max     0.985945   0.973980   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.961073


In [14]:
model.fit(X_updated_w5_scaled, Y_updated_w5_scaled)

betas = [0.001, 0.005, 0.01, 0.05, 0.10, 0.15, 0.25]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'x7': best_x_raw[6],
        'x8': best_x_raw[7],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)

pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

print("--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 8: 8D LOCAL MAXIMA OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5       x6       x7       x8  Pred Mean  Uncertainty
0.001000 0.139604 0.155475 0.106466 0.180774 1.000000 0.469253 0.204475 0.299337   9.966154     0.017620
0.005000 0.139259 0.155499 0.106124 0.181039 1.000000 0.469391 0.204401 0.299336   9.966154     0.017814
0.010000 0.138823 0.155534 0.105690 0.181358 1.000000 0.469560 0.204306 0.299338   9.966152     0.018058
0.050000 0.135097 0.155988 0.102093 0.183857 1.000000 0.471071 0.203444 0.299336   9.966088     0.020150
0.100000 0.129835 0.157062 0.097247 0.186715 1.000000 0.473304 0.202034 0.299337   9.965863     0.023132
0.150000 0.123921 0.158753 0.092044 0.189279 1.000000 0.475848 0.200205 0.299337   9.965432     0.026565
0.250000 0.110293 0.163768 0.080658 0.193554 1.000000 0.481707 0.195225 0.299337   9.963750     0.034888

--- Optimized Kernel Parameters ---
Matern(length_scale=[4.71, 7.08, 3.75, 8.9, 16.9, 8.72, 5.67, 1e+05], nu=2

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 6 submission

Next point to query [0.139259 0.155499 0.106124 0.181039 1.000000 0.469391 0.204401 0.299336] beta = 0.005

* The improvement is small ~0,05 in absolute terms, with the next round it's expected to be ~0.01 
* Lowered the beta even further where the beta sweep predicted means show that the performance is about to plateau 
* If the function behaves as the model predicts - try a neural network approach

In [15]:
# ---------------------------------------------------
# WEEK6: NEW DATA HERE
# ---------------------------------------------------

w6_new_input = 	[0.139259, 0.155499, 0.106124, 0.181039, 1.000000, 0.469391, 0.204401, 0.299336]
w6_new_output = 9.9641980806154

w6_new_X = np.atleast_2d(w6_new_input)
w6_new_Y = np.atleast_1d(w6_new_output)

X_updated_w6 = np.vstack((X_updated_w5, w6_new_X))
Y_updated_w6 = np.append(Y_updated_w5, w6_new_Y)

X_updated_w6_scaled = scaler_x.fit_transform(X_updated_w6)
Y_updated_w6_scaled = scaler_y.fit_transform(Y_updated_w6.reshape(-1, 1))

df = pd.DataFrame(X_updated_w6, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w6

# print(df.describe())
print(df.head(46))

          x1        x2        x3        x4        x5        x6        x7        x8    target
0   0.604994  0.292215  0.908453  0.355506  0.201669  0.575338  0.310311  0.734281  7.398721
1   0.178007  0.566223  0.994862  0.210325  0.320153  0.707909  0.635384  0.107132  7.005227
2   0.009077  0.811626  0.520520  0.075687  0.265112  0.091652  0.592415  0.367320  8.459482
3   0.506028  0.653730  0.363411  0.177981  0.093728  0.197425  0.755827  0.292472  8.284008
4   0.359909  0.249076  0.495997  0.709215  0.114987  0.289207  0.557295  0.593882  8.606117
5   0.778818  0.003419  0.337983  0.519528  0.820907  0.537247  0.551347  0.660032  8.541748
6   0.908649  0.062250  0.238260  0.766604  0.132336  0.990244  0.688068  0.742496  7.327435
7   0.586371  0.880736  0.745021  0.546035  0.009649  0.748992  0.230907  0.097916  7.299872
8   0.761137  0.854672  0.382124  0.337352  0.689708  0.309853  0.631380  0.041956  7.957875
9   0.984933  0.699506  0.998885  0.180148  0.580143  0.231087  0.4908

In [22]:
# exploitation_df = df[df['target'] > 9.0].copy()
# exploitation_df.reset_index(drop=True, inplace=True)

# # Define our feature sets based on the model diagnostics
# features_8d = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']
# features_7d = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7']       # Dropping the flat x8
# features_6d = ['x1', 'x2', 'x3', 'x4', 'x6', 'x7']             # Dropping x8 and the pegged x5
# features_5d = ['x2', 'x3', 'x4', 'x6', 'x7']                   # Dropping x8, x5, and the newly flat x1

# y = exploitation_df['target'].values

# def create_gp_model(n_dims):
#     """Dynamically creates a GP with the correct number of length scales."""
#     initial_length_scales = np.array([1.0] * n_dims)
#     kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
#              WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-20, 1e1))
    
#     return GaussianProcessRegressor(
#         kernel=kernel, 
#         n_restarts_optimizer=50,  
#         random_state=42
#     )

# # Setup models
# models = {
#     'Model A (8D - Full)': (create_gp_model(8), features_8d),
#     'Model B (7D - Drop x8)': (create_gp_model(7), features_7d),
#     'Model C (6D - Drop x8, x5)': (create_gp_model(6), features_6d),
#     'Model D (5D - Drop x8, x5, x1)': (create_gp_model(5), features_5d)
# }

# results = []
# loo = LeaveOneOut()

# for model_name, (model, features) in models.items():
#     X = exploitation_df[features].values
#     y_true = []
#     y_pred_mean = []
#     y_pred_std = []
    
#     # Run LOOCV
#     for train_index, test_index in loo.split(X):
#         X_train, X_test = X[train_index], X[test_index]
#         y_train, y_test = y[train_index], y[test_index]
        
#         model.fit(X_train, y_train)
#         mean, std = model.predict(X_test, return_std=True)
        
#         y_true.append(y_test[0])
#         y_pred_mean.append(mean[0])
#         y_pred_std.append(std[0])
        
#     # Calculate overall metrics for this dimensionality
#     mse = mean_squared_error(y_true, y_pred_mean)
#     mae = mean_absolute_error(y_true, y_pred_mean)
#     avg_uncertainty = np.mean(y_pred_std)
    
#     # Fit once on the full exploitation set to get the Log Marginal Likelihood
#     model.fit(X, y)
#     lml = model.log_marginal_likelihood_value_
    
#     results.append({
#         'Model Configuration': model_name,
#         'Dimensions': len(features),
#         'MSE': mse,
#         'MAE': mae,
#         'Avg Target Uncertainty (Std)': avg_uncertainty,
#         'Log Marginal Likelihood': lml
#     })

# # Output the comparison
# comparison_df = pd.DataFrame(results)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)

# print("\n--- DIMENSIONALITY EVALUATION RESULTS ---")
# print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

In [17]:

# Use the raw Y array to find where the target is > 9.0, then apply that mask to the scaled arrays
mask = Y_updated_w6 > 9.0
X_exploit_scaled = X_updated_w6_scaled[mask]
Y_exploit_scaled = Y_updated_w6_scaled[mask]

active_indices = [1, 2, 3, 5, 6]
X_train_5d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

# Train the highly focused 5D GP
initial_length_scales = np.array([1.0] * 5)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-20, 1e1))

model_5d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_5d.fit(X_train_5d, y_train)

# Define Pure Mean Maximization (by minimizing the negative mean)
def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

# Find the best known point in the active dimensions to use as our starting guess (x0)
best_idx = np.argmax(y_train)
x0_5d = X_train_5d[best_idx]
raw_lower_bound = [[0, 0, 0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_5d = []
for idx in active_indices:
    bounds_5d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_5d,
    args=(model_5d,),
    method='L-BFGS-B',
    bounds=bounds_5d
)

best_x_5d = res.x
predicted_max_mean_scaled = -res.fun

# Reconstruct the full 8D array in the SCALED space
best_full_row_scaled = X_exploit_scaled[best_idx]
final_8d_scaled = np.zeros(8)
final_8d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen at best known
final_8d_scaled[1] = best_x_5d[0]             # x2: Optimized
final_8d_scaled[2] = best_x_5d[1]             # x3: Optimized
final_8d_scaled[3] = best_x_5d[2]             # x4: Optimized
final_8d_scaled[4] = best_full_row_scaled[4]  # x5: Frozen accurately at its peak
final_8d_scaled[5] = best_x_5d[3]             # x6: Optimized
final_8d_scaled[6] = best_x_5d[4]             # x7: Optimized
final_8d_scaled[7] = best_full_row_scaled[7]  # x8: Frozen at best known

# Convert back to physical parameters using your scalers
final_8d_raw = scaler_x.inverse_transform(final_8d_scaled.reshape(1, -1))[0]

# Inverse transform the predicted mean target
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

# 8. Output the final recommendation
print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
feature_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']
for name, val in zip(feature_names, final_8d_raw):
    print(f"{name}: {val:.6f}")
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
x1: 0.139259
x2: 1.000000
x3: 0.106115
x4: 0.186278
x5: 1.000000
x6: 0.000000
x7: 0.000000
x8: 0.299336

Predicted Target Mean: 10.277593


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 7 submission

Next point to query [0.139259 1.000000 0.106115 0.186278 1.000000 0.000000 0.000000 0.299336] 

* Analyzed baseline 8D Gaussian Process model's optimized kernel parameters and found that x8​ had a length scale hitting the upper bound (100,000), meaning it had completely flatlined in the local region.

* Proposed Alternative Strategies: Evaluated how to move past a performance plateau near 9.96 using three potential approaches: Dimensionality Freezing, Trust Region Exploitation, and Pure Mean Maximization.

* Tried dimensionality freezing by running Leave-One-Out Cross-Validation exclusively on the high-performing exploitation region (targets >9.0) across four configurations:

    Model A (8D - Full) → MAE: 0.149726, LML: -34.957065

    Model B (7D - Drop x8​) → MAE: 0.121834, LML: -34.957034

    Model C (6D - Drop x8​,x5​) → MAE: 0.046427, LML: -35.284042

    Model D (5D - Drop x8​,x5​,x1​) → MAE: 0.046427, LML: -35.284042

* Found Model D to be the best performing and decided to use it as the model for generating the next query point. 

* Ranpure mean maximization (L-BFGS-B optimizer on the negative predicted mean) and generated a new physical query point with an unprecedented predicted target mean of 10.277593.

In [18]:
# ---------------------------------------------------
# WEEK7: NEW DATA HERE
# ---------------------------------------------------

w7_new_input = 	[0.139259, 1.000000, 0.106115, 0.186278, 1.000000, 0.000000, 0.000000, 0.299336]
w7_new_output = 8.9123500047894

w7_new_X = np.atleast_2d(w7_new_input)
w7_new_Y = np.atleast_1d(w7_new_output)

X_updated_w7 = np.vstack((X_updated_w6, w7_new_X))
Y_updated_w7 = np.append(Y_updated_w6, w7_new_Y)

X_updated_w7_scaled = scaler_x.fit_transform(X_updated_w7)
Y_updated_w7_scaled = scaler_y.fit_transform(Y_updated_w7.reshape(-1, 1))

df = pd.DataFrame(X_updated_w7, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w7

print(df.describe())
# print(df.head(47))

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  47.000000  47.000000  47.000000  47.000000  47.000000  47.000000  47.000000  47.000000  47.000000
mean    0.481050   0.441458   0.461591   0.381169   0.546885   0.468271   0.518186   0.485279   8.100441
std     0.314318   0.318126   0.291835   0.270124   0.320333   0.279695   0.281880   0.283085   1.128329
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.000000   0.000000   0.000000   5.592193
25%     0.177078   0.137413   0.225905   0.137333   0.256311   0.248903   0.247344   0.293995   7.319164
50%     0.472071   0.458759   0.382124   0.355506   0.580143   0.445117   0.553779   0.476462   8.042213
75%     0.778582   0.676618   0.724607   0.626163   0.808497   0.706325   0.759155   0.722286   8.944452
max     0.985945   1.000000   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.964198


In [19]:

# Use the raw Y array to find where the target is > 7.0, then apply that mask to the scaled arrays
mask = Y_updated_w7 > 7.0
X_exploit_scaled = X_updated_w7_scaled[mask]
Y_exploit_scaled = Y_updated_w7_scaled[mask]

active_indices = [1, 2, 3, 5, 6]
X_train_5d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

# Train the highly focused 5D GP
initial_length_scales = np.array([1.0] * 5)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-20, 1e1))

model_5d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_5d.fit(X_train_5d, y_train)

# Define Pure Mean Maximization (by minimizing the negative mean)
def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

# Find the best known point in the active dimensions to use as our starting guess (x0)
best_idx = np.argmax(y_train)
x0_5d = X_train_5d[best_idx]
raw_lower_bound = [[0, 0, 0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_5d = []
for idx in active_indices:
    bounds_5d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_5d,
    args=(model_5d,),
    method='L-BFGS-B',
    bounds=bounds_5d
)

best_x_5d = res.x
predicted_max_mean_scaled = -res.fun

# Reconstruct the full 8D array in the SCALED space
best_full_row_scaled = X_exploit_scaled[best_idx]
final_8d_scaled = np.zeros(8)
final_8d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen at best known
final_8d_scaled[1] = best_x_5d[0]             # x2: Optimized
final_8d_scaled[2] = best_x_5d[1]             # x3: Optimized
final_8d_scaled[3] = best_x_5d[2]             # x4: Optimized
final_8d_scaled[4] = best_full_row_scaled[4]  # x5: Frozen accurately at its peak
final_8d_scaled[5] = best_x_5d[3]             # x6: Optimized
final_8d_scaled[6] = best_x_5d[4]             # x7: Optimized
final_8d_scaled[7] = best_full_row_scaled[7]  # x8: Frozen at best known

# Convert back to physical parameters using your scalers
final_8d_raw = scaler_x.inverse_transform(final_8d_scaled.reshape(1, -1))[0]

# Inverse transform the predicted mean target
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_8d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.139259, 0.086521, 0.068065, 0.114160, 1.000000, 0.436247, 0.182049, 0.299336]

Predicted Target Mean: 10.009226


### Week 8 submission

Next point to query [0.139259, 0.086521, 0.068065, 0.114160, 1.000000, 0.436247, 0.182049, 0.299336]

* The performance didn't match the predicted target mean 
* Worth including in the next run a measure of uncertainty 
* Changed the filter from 9 to 7 (to increase the points to train) - with the hope the model is more accurate

In [20]:
# ---------------------------------------------------
# WEEK8: NEW DATA HERE
# ---------------------------------------------------

w8_new_input = 	[0.139259, 0.086521, 0.068065, 0.114160, 1.000000, 0.436247, 0.182049, 0.299336]
w8_new_output = 9.9463467342214

w8_new_X = np.atleast_2d(w8_new_input)
w8_new_Y = np.atleast_1d(w8_new_output)

X_updated_w8 = np.vstack((X_updated_w7, w8_new_X))
Y_updated_w8 = np.append(Y_updated_w7, w8_new_Y)

X_updated_w8_scaled = scaler_x.fit_transform(X_updated_w8)
Y_updated_w8_scaled = scaler_y.fit_transform(Y_updated_w8.reshape(-1, 1))

df = pd.DataFrame(X_updated_w8, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w8

print(df.describe())
# print(df.head(48))

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  48.000000  48.000000  48.000000  48.000000  48.000000  48.000000  48.000000  48.000000  48.000000
mean    0.473929   0.434063   0.453393   0.375606   0.556324   0.467604   0.511184   0.481405   8.138898
std     0.314845   0.318866   0.294248   0.270000   0.323585   0.276743   0.283054   0.281340   1.147617
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.000000   0.000000   0.000000   5.592193
25%     0.173256   0.116857   0.218616   0.132574   0.260711   0.250581   0.239557   0.294756   7.323299
50%     0.459583   0.448247   0.379438   0.346429   0.619797   0.440682   0.552563   0.474430   8.101098
75%     0.778464   0.665174   0.722896   0.619669   0.837553   0.706163   0.757491   0.719124   8.985684
max     0.985945   1.000000   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.964198


In [21]:

# Use the raw Y array to find where the target is > 7.0, then apply that mask to the scaled arrays
mask = Y_updated_w8 > 7.0
X_exploit_scaled = X_updated_w8_scaled[mask]
Y_exploit_scaled = Y_updated_w8_scaled[mask]

active_indices = [1, 2, 3, 5, 6]
X_train_5d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 5)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-20, 1e1))

model_5d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_5d.fit(X_train_5d, y_train)

# Define Pure Mean Maximization (by minimizing the negative mean)
def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

# Find the best known point in the active dimensions to use as our starting guess (x0)
best_idx = np.argmax(y_train)
x0_5d = X_train_5d[best_idx]
raw_lower_bound = [[0, 0, 0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_5d = []
for idx in active_indices:
    bounds_5d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_5d,
    args=(model_5d,),
    method='L-BFGS-B',
    bounds=bounds_5d
)

best_x_5d = res.x
predicted_max_mean_scaled = -res.fun

# Reconstruct the full 8D array in the SCALED space
best_full_row_scaled = X_exploit_scaled[best_idx]
final_8d_scaled = np.zeros(8)
final_8d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen at best known
final_8d_scaled[1] = best_x_5d[0]             # x2: Optimized
final_8d_scaled[2] = best_x_5d[1]             # x3: Optimized
final_8d_scaled[3] = best_x_5d[2]             # x4: Optimized
final_8d_scaled[4] = best_full_row_scaled[4]  # x5: Frozen accurately at its peak
final_8d_scaled[5] = best_x_5d[3]             # x6: Optimized
final_8d_scaled[6] = best_x_5d[4]             # x7: Optimized
final_8d_scaled[7] = best_full_row_scaled[7]  # x8: Frozen at best known

# Convert back to physical parameters using your scalers
final_8d_raw = scaler_x.inverse_transform(final_8d_scaled.reshape(1, -1))[0]

# Inverse transform the predicted mean target
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_8d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.139259, 0.145008, 0.104970, 0.158781, 1.000000, 0.495204, 0.154003, 0.299336]

Predicted Target Mean: 9.977519


### Week 9 submission

Next point to query [0.139259, 0.145008, 0.104970, 0.158781, 1.000000, 0.495204, 0.154003, 0.299336]

* Model is working

In [24]:
# ---------------------------------------------------
# WEEK9: NEW DATA HERE
# ---------------------------------------------------

w9_new_input = [0.139259, 0.145008, 0.104970, 0.158781, 1.000000, 0.495204, 0.154003, 0.299336]
w9_new_output = 9.9616415993894

w9_new_X = np.atleast_2d(w9_new_input)
w9_new_Y = np.atleast_1d(w9_new_output)

X_updated_w9 = np.vstack((X_updated_w8, w9_new_X))
Y_updated_w9 = np.append(Y_updated_w8, w9_new_Y)

X_updated_w9_scaled = scaler_x.fit_transform(X_updated_w9)
Y_updated_w9_scaled = scaler_y.fit_transform(Y_updated_w9.reshape(-1, 1))

df = pd.DataFrame(X_updated_w9, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'])
df['target'] = Y_updated_w9

print(df.describe())
# print(df.head(39))

              x1         x2         x3         x4         x5         x6         x7         x8     target
count  49.000000  49.000000  49.000000  49.000000  49.000000  49.000000  49.000000  49.000000  49.000000
mean    0.467099   0.428164   0.446282   0.371181   0.565379   0.468167   0.503894   0.477689   8.176096
std     0.315195   0.318217   0.295391   0.268962   0.326410   0.273873   0.284700   0.279607   1.165072
min     0.009077   0.003419   0.022929   0.000000   0.009649   0.000000   0.000000   0.000000   5.592193
25%     0.164574   0.119328   0.216005   0.134396   0.265112   0.252259   0.237647   0.295518   7.327435
50%     0.447096   0.437736   0.376751   0.337352   0.659452   0.445117   0.551347   0.472398   8.159983
75%     0.778345   0.653730   0.721184   0.613175   0.887490   0.706000   0.755827   0.715962   9.013075
max     0.985945   1.000000   0.998885   0.902986   1.000000   1.000000   0.992914   0.988755   9.964198


In [25]:

# Use the raw Y array to find where the target is > 7.0, then apply that mask to the scaled arrays
mask = Y_updated_w9 > 7.0
X_exploit_scaled = X_updated_w9_scaled[mask]
Y_exploit_scaled = Y_updated_w9_scaled[mask]

active_indices = [1, 2, 3, 5, 6]
X_train_5d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 5)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-20, 1e1))

model_5d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_5d.fit(X_train_5d, y_train)

# Define Pure Mean Maximization (by minimizing the negative mean)
def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

# Find the best known point in the active dimensions to use as our starting guess (x0)
best_idx = np.argmax(y_train)
x0_5d = X_train_5d[best_idx]
raw_lower_bound = [[0, 0, 0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_5d = []
for idx in active_indices:
    bounds_5d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_5d,
    args=(model_5d,),
    method='L-BFGS-B',
    bounds=bounds_5d
)

best_x_5d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_8d_scaled = np.zeros(8)
final_8d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen at best known
final_8d_scaled[1] = best_x_5d[0]             # x2: Optimized
final_8d_scaled[2] = best_x_5d[1]             # x3: Optimized
final_8d_scaled[3] = best_x_5d[2]             # x4: Optimized
final_8d_scaled[4] = best_full_row_scaled[4]  # x5: Frozen accurately at its peak
final_8d_scaled[5] = best_x_5d[3]             # x6: Optimized
final_8d_scaled[6] = best_x_5d[4]             # x7: Optimized
final_8d_scaled[7] = best_full_row_scaled[7]  # x8: Frozen at best known

final_8d_raw = scaler_x.inverse_transform(final_8d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_8d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.139259, 0.095799, 0.100429, 0.169243, 1.000000, 0.440904, 0.211885, 0.299336]

Predicted Target Mean: 9.974520


### Week 10 submission

Next point to query [0.139259, 0.095799, 0.100429, 0.169243, 1.000000, 0.440904, 0.211885, 0.299336]

* Close to max